========================================================

># <strong>PROJETO DATA MAJOR</strong> - Tópicos de Banco de Dados (IESB 2026)
>## <strong>DataSus - SINAN</strong>
>## <strong>Tema: </strong> Análise de padrões de hospitalização em casos de dengue com base no perfil clínico, epidemiológico e social dos pacientes utilizando dados do SINAN
>### + Análise estatística seguida de clusterização k-means

<br>
Documentação SINAN: https://pysus.readthedocs.io/en/latest/databases/SINAN.html
<br>
<br>
================================================================

<strong>Docente: </strong>Rodrigo Gonçalves<br>

<strong>Integrantes do Grupo:</strong>
- Beatriz Brito do Rosário - Extração
- Davi de Souza Lopes - Transform
- Lucas de Siqueira Cavalcanti - Load
- Lucas Montalvão Ramires - Classificação


---
### *Acesso ao Google Drive e Pastas do Projeto*

---



In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

base_path      = '/content/drive/MyDrive/Topicos_BD'
raw_sinan      = os.path.join(base_path, 'raw', 'sinan')
raw_ibge       = os.path.join(base_path, 'raw', 'ibge')
processed_path = os.path.join(base_path, 'processed')
parquet_path   = os.path.join(base_path, 'parquet')

for path in [raw_sinan, raw_ibge, processed_path, parquet_path]:
    os.makedirs(path, exist_ok=True)

Mounted at /content/drive
Estrutura de pastas criada:
raw/sinan -> /content/drive/MyDrive/Topicos_BD/raw/sinan
raw/ibge  -> /content/drive/MyDrive/Topicos_BD/raw/ibge
processed -> /content/drive/MyDrive/Topicos_BD/processed
parquet   -> /content/drive/MyDrive/Topicos_BD/parquet


---
### *Instalação e Importação*
---

In [4]:
!pip install -q pysus matplotlib pandas pyarrow pyftpdlib pyspark

In [5]:
from pysus import SINAN
from pyspark.sql import SparkSession
import pandas as pd
import glob
import os
import requests

---
## *Extração — SINAN Dengue 2022 + 2023*

**Responsável:** Beatriz Brito

**Fonte:** Servidor FTP público do DATASUS  
**Código da doença:** `DENG` (Dengue)  
**Período:** 2022 e 2023  
**Formato baixado:** `.DBC` → convertido automaticamente para `.parquet` pelo PySUS

In [6]:
def extract_sinan(local_dir):
    """
    Baixa os arquivos de dengue do SINAN via FTP do DATASUS.
    O PySUS converte automaticamente .DBC para .parquet.
    Retorna a lista de arquivos .parquet gerados.
    """
    sinan = SINAN().load()

    # Anos de 2022 e 2023
    arquivos = (sinan.get_files('DENG', year=2022) +
                sinan.get_files('DENG', year=2023))

    print('\nIniciando download...')
    sinan.download(arquivos, local_dir=local_dir)

    # Localiza todos os parquets gerados (podem estar em subpastas)
    parquet_files = glob.glob(os.path.join(local_dir, '**', '*.parquet'), recursive=True)
    print(f'\nDownload concluído. {len(parquet_files)} arquivo(s) .parquet em {local_dir}')

    return parquet_files

parquet_files = extract_sinan(raw_sinan)


Iniciando download...


DENGBR23.parquet: 100%|██████████| 2.68M/2.68M [04:46<00:00, 9.37kB/s]


Download concluído. 104 arquivo(s) .parquet em /content/drive/MyDrive/Topicos_BD/raw/sinan


#### Leitura com PySpark e seleção de colunas

In [7]:
# sessao spark
spark = SparkSession.builder \
    .appName('SINAN-Dengue-Extract') \
    .getOrCreate()

if not parquet_files:
    raise FileNotFoundError('Nenhum .parquet encontrado. Execute o download primeiro.')

df_sinan = spark.read.parquet(*parquet_files)

print(f'Total de registros : {df_sinan.count()}')
print(f'Total de colunas   : {len(df_sinan.columns)}')
print(f'Colunas disponíveis: {df_sinan.columns}')

Total de registros : 6102102
Total de colunas   : 121
Colunas disponíveis: ['TP_NOT', 'ID_AGRAVO', 'DT_NOTIFIC', 'SEM_NOT', 'NU_ANO', 'SG_UF_NOT', 'ID_MUNICIP', 'ID_REGIONA', 'ID_UNIDADE', 'DT_SIN_PRI', 'SEM_PRI', 'ANO_NASC', 'NU_IDADE_N', 'CS_SEXO', 'CS_GESTANT', 'CS_RACA', 'CS_ESCOL_N', 'SG_UF', 'ID_MN_RESI', 'ID_RG_RESI', 'ID_PAIS', 'DT_INVEST', 'ID_OCUPA_N', 'FEBRE', 'MIALGIA', 'CEFALEIA', 'EXANTEMA', 'VOMITO', 'NAUSEA', 'DOR_COSTAS', 'CONJUNTVIT', 'ARTRITE', 'ARTRALGIA', 'PETEQUIA_N', 'LEUCOPENIA', 'LACO', 'DOR_RETRO', 'DIABETES', 'HEMATOLOG', 'HEPATOPAT', 'RENAL', 'HIPERTENSA', 'ACIDO_PEPT', 'AUTO_IMUNE', 'DT_CHIK_S1', 'DT_CHIK_S2', 'DT_PRNT', 'RES_CHIKS1', 'RES_CHIKS2', 'RESUL_PRNT', 'DT_SORO', 'RESUL_SORO', 'DT_NS1', 'RESUL_NS1', 'DT_VIRAL', 'RESUL_VI_N', 'DT_PCR', 'RESUL_PCR_', 'SOROTIPO', 'HISTOPA_N', 'IMUNOH_N', 'HOSPITALIZ', 'DT_INTERNA', 'UF', 'MUNICIPIO', 'TPAUTOCTO', 'COUFINF', 'COPAISINF', 'COMUNINF', 'CLASSI_FIN', 'CRITERIO', 'DOENCA_TRA', 'CLINC_CHIK', 'EVOLUCAO', 'DT

In [31]:
# colunas (perfil clínico + epidemiológico + variável alvo)
COLUNAS_INTERESSE = [
    # variável alvo
    'HOSPITALIZ',
    # perfil epidemiológico/social
    'CS_SEXO', 'NU_IDADE_N', 'ID_MUNICIP', 'ID_OCUPA_N', 'NU_ANO',
    # sinais clínicos
    'FEBRE', 'MIALGIA', 'CEFALEIA', 'VOMITO', 'NAUSEA',
    'DOR_RETRO', 'DOR_COSTAS', 'EXANTEMA', 'LEUCOPENIA',
    # comorbidades
    'DIABETES', 'HIPERTENSA', 'RENAL', 'HEPATOPAT', 'HEMATOLOG',
    # classificação e desfecho
    'CLASSI_FIN', 'EVOLUCAO',
]

df_sinan = df_sinan.select(COLUNAS_INTERESSE)
df_sinan.show(10)

+----------+-------+----------+----------+----------+------+-----+-------+--------+------+------+---------+----------+--------+----------+--------+----------+-----+---------+---------+----------+--------+
|HOSPITALIZ|CS_SEXO|NU_IDADE_N|ID_MUNICIP|ID_OCUPA_N|NU_ANO|FEBRE|MIALGIA|CEFALEIA|VOMITO|NAUSEA|DOR_RETRO|DOR_COSTAS|EXANTEMA|LEUCOPENIA|DIABETES|HIPERTENSA|RENAL|HEPATOPAT|HEMATOLOG|CLASSI_FIN|EVOLUCAO|
+----------+-------+----------+----------+----------+------+-----+-------+--------+------+------+---------+----------+--------+----------+--------+----------+-----+---------+---------+----------+--------+
|         2|      M|      4035|    351750|    414105|  2022|    1|      2|       2|     2|     1|        1|         1|       2|         2|       2|         2|    2|        2|        2|        10|       1|
|         2|      M|      4033|    353250|          |  2022|    1|      1|       1|     2|     2|        1|         1|       2|         2|       2|         2|    2|        2|      

In [15]:
# salva o SINAN consolidado em raw/sinan/
output_sinan = os.path.join(raw_sinan, 'sinan_dengue_raw')
df_sinan.write.mode('overwrite').parquet(output_sinan)

tamanho_mb = sum(
    os.path.getsize(f)
    for f in glob.glob(os.path.join(output_sinan, '*.parquet'))
) / 1e6

print(f'Dados do SINAN salvo em: {output_sinan}')
print(f'Registros: {df_sinan.count()}')

# Conferir se tamanho confere com o requisito de 30mb - 60mb
print(f'Tamanho estimado: {tamanho_mb:.1f} MB')

Dados do SINAN salvo em: /content/drive/MyDrive/Topicos_BD/raw/sinan/sinan_dengue_raw
Registros: 6102102
Tamanho estimado: 36.4 MB


# Extração — IBGE API (dados de municípios)
Fonte: API pública do IBGE — servicodados.ibge.gov.br
<br></br>
Motivo: O campo ID_MUNICIP do SINAN é o código IBGE do município. Com a API conseguimos adicionar nome, UF, região e população - informações essenciais para o perfil epidemiológico e para calcular taxas de hospitalização por município.

In [26]:
def extract_ibge_municipios(local_dir):
    url = 'https://servicodados.ibge.gov.br/api/v1/localidades/municipios'

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    dados = response.json()
    print(f'Itens recebidos: {len(dados)}')

    registros = []

    for m in dados:
        microrregiao  = m.get('microrregiao') or {}
        mesorregiao   = microrregiao.get('mesorregiao') or {}
        uf            = mesorregiao.get('UF') or {}
        regiao        = uf.get('regiao') or {}

        registros.append({
            'id_municipio'  : str(m.get('id', '')),
            'nome_municipio': m.get('nome', ''),
            'uf_codigo'     : str(uf.get('id', '')),
            'uf_sigla'      : uf.get('sigla', ''),
            'uf_nome'       : uf.get('nome', ''),
            'regiao'        : regiao.get('nome', ''),
        })


    print(f'Municípios processados: {len(registros)}')

    df = pd.DataFrame(registros)

    # pra ajudar no transofrm pois no sinann o id so tem 6 digitos
    df['id_municipio_sinan'] = df['id_municipio'].str[:6]

    output = os.path.join(local_dir, 'ibge_municipios.parquet')
    df.to_parquet(output, index=False)

    print(f'Salvo em: {output}')
    return df

df_municipios = extract_ibge_municipios(raw_ibge)
df_municipios.head()

Itens recebidos: 5571
Municípios processados: 5571
Salvo em: /content/drive/MyDrive/Topicos_BD/raw/ibge/ibge_municipios.parquet


,id_municipio,nome_municipio,uf_codigo,uf_sigla,uf_nome,regiao,id_municipio_sinan
0,1100015,Alta Floresta D'Oeste,11,RO,Rondônia,Norte,110001
1,1100023,Ariquemes,11,RO,Rondônia,Norte,110002
2,1100031,Cabixi,11,RO,Rondônia,Norte,110003
3,1100049,Cacoal,11,RO,Rondônia,Norte,110004
4,1100056,Cerejeiras,11,RO,Rondônia,Norte,110005


In [27]:
# distribuição de municípios por região
print('Municípios por região:')
print(df_municipios.groupby('regiao')['nome_municipio'].count().sort_values(ascending=False).to_string())

print(f'\nTotal de municípios extraídos: {len(df_municipios)}')
print(f'Total de UFs: {df_municipios["uf_sigla"].nunique()}')

Municípios por região:
regiao
Nordeste        1794
Sudeste         1668
Sul             1191
Centro-Oeste     467
Norte            450
                   1

Total de municípios extraídos: 5571
Total de UFs: 28


In [50]:
# tamanho dos arquivos salvos
arquivos_sinan = glob.glob(os.path.join(output_sinan, '*.parquet'))
tamanho_mb = sum(os.path.getsize(f) for f in arquivos_sinan) / 1e6

print('=' * 30)
print('      RESUMO DO EXTRACT')
print('=' * 30)

print('\n[SINAN — DATASUS FTP]')
print(f'  Período    : 2022 e 2023')
print(f'  Registros  : {df_sinan.count():,}')
print(f'  Colunas    : {len(df_sinan.columns)}')
print(f'  Tamanho    : {tamanho_mb:.1f} MB')
print(f'  Salvo em   : raw/sinan/sinan_dengue_raw/')

print('\n[IBGE — API pública]')
print(f'  Municípios : {len(df_municipios):,}')
print(f'  UFs        : {df_municipios["uf_sigla"].nunique()}')
print(f'  Salvo em   : raw/ibge/ibge_municipios.parquet')

print('=' * 52)

      RESUMO DO EXTRACT

[SINAN — DATASUS FTP]
  Período    : 2022 e 2023
  Registros  : 6,102,102
  Colunas    : 22
  Tamanho    : 36.4 MB
  Salvo em   : raw/sinan/sinan_dengue_raw/

[IBGE — API pública]
  Municípios : 5,571
  UFs        : 28
  Salvo em   : raw/ibge/ibge_municipios.parquet
